# Patient subscribed lockers / locker details test

Runs a real PHR login (OTP-based) against the ABDM sandbox, then calls:
- `GET /api/hiecm/subscription-requests/v3/patients/lockers` (patient-subscribed-lockers)
- `GET /api/hiecm/subscription-requests/v3/patients/lockers/{lockerId}` (patient-locker-details-by-locker-id)

Both from the Postman collection's **PHR &rarr; Subscription and Health locker** folder.

## Findings so far (2026-09-10) — read before re-running

Both endpoints reject **every** patient token tried with `{"code": "ABDM-1066: ", "message": "Invalid JWT token"}`, across:

- `profile/login` (loginHint=`mobile`) &rarr; OTP-verify T-Token
- `profile/login` (loginHint=`mobile`) &rarr; final X-Token (after the `verify_user` account-selection step)
- `phr/web/login/abha` (ABHA Address login) &rarr; direct token
- `profile/login` (loginHint=`abha-number`) &rarr; direct token

Also tried, with the gateway bearer token minted from **both** `CLIENT_ID`/`CLIENT_SECRET` (old bridge, `SBXID_046112`) and `PHR_CLIENT_ID`/`PHR_CLIENT_SECRET` (new bridge, `SBXID_073333`) — no difference, identical rejection either way.

One data point that changes the error shape (not yet explained): dropping the `X-CM-ID` header entirely (while still sending `Authorization` + `X-AUTH-TOKEN`) changes the response from `ABDM-1066 Invalid JWT token` to a generic `500 {"code":"900900","message":"Unclassified Authentication Failure", ...}` — so `X-CM-ID` is clearly read/consumed somewhere in the chain, but its presence doesn't fix the underlying rejection.

Aayush has confirmed the test patient (`poojaanchaliya@sbx` / Pooja Rameshkumar) **is** already subscribed to the government health locker, so this is unlikely to be the same "not provisioned" story as the `ABDM-1040 Invalid HIU ID` subscription-init issue (see the `project-hiu-role-not-provisioned` memory) — something about how `X-AUTH-TOKEN` is being sent/validated here specifically is still wrong, not yet root-caused. This notebook exists so the next hypothesis can be tried in under a minute, without repeating the login legwork.

## Setup

In [ ]:
import sys
sys.path.insert(0, r"C:\Users\hp\Desktop\Aayush\repo")

import json
import uuid

import requests

from server.config import (
    CLIENT_ID, CLIENT_SECRET, GATEWAY_BASE_URL, HIECM_BASE_URL,
    PHR_CLIENT_ID, PHR_CLIENT_SECRET, X_CM_ID,
)
from server.utils import generate_request_id, generate_timestamp
from server.abha import request_otp, verify_otp
from server.crypto import get_public_certificate, encrypt_value

print("CLIENT_ID (old bridge):", CLIENT_ID)
print("PHR_CLIENT_ID (new bridge) set:", bool(PHR_CLIENT_ID))

## Which bridge identity to use for the gateway token

Set `TEST_CLIENT_ID`/`TEST_CLIENT_SECRET` — defaults to the old bridge (`CLIENT_ID`). Swap to `PHR_CLIENT_ID`/`PHR_CLIENT_SECRET` to test the new one instead. Both were tried already (see the findings cell above) with no difference in outcome, but this is here for whichever comes up next.

In [ ]:
TEST_CLIENT_ID = CLIENT_ID          # <-- swap to PHR_CLIENT_ID to test the new bridge instead
TEST_CLIENT_SECRET = CLIENT_SECRET  # <-- swap to PHR_CLIENT_SECRET to match

if not TEST_CLIENT_ID or not TEST_CLIENT_SECRET:
    raise SystemExit("TEST_CLIENT_ID / TEST_CLIENT_SECRET not set -- fill in repo/.env first.")

## Step 1 — Gateway token

Mints a fresh token AS `TEST_CLIENT_ID` -- not cached, not shared with `server.utils.get_gateway_token()`. Re-run this cell to get a fresh token if it's been a while (`expiresIn` below).

In [ ]:
token_response = requests.post(
    url=f"{GATEWAY_BASE_URL}/sessions",
    json={"clientId": TEST_CLIENT_ID, "clientSecret": TEST_CLIENT_SECRET, "grantType": "client_credentials"},
    headers={
        "Content-Type": "application/json",
        "REQUEST-ID": generate_request_id(),
        "TIMESTAMP": generate_timestamp(),
        "X-CM-ID": X_CM_ID,
    },
    timeout=30,
)
print("Token generation status:", token_response.status_code)
if token_response.status_code != 200:
    print("Body:", token_response.text)
    raise SystemExit("Token generation failed -- fix this before continuing.")

_token_data = token_response.json()
gateway_token = _token_data["accessToken"]
print("expiresIn (seconds):", _token_data.get("expiresIn"))
# Deliberately not printing gateway_token itself -- it's a live bearer credential.

## Step 2 — PHR login (OTP-based) — get a patient X-AUTH-TOKEN

Defaults to the `profile/login` / `loginHint=abha-number` variant (Flow 5) -- the last one tried, returns a direct usable token with no extra `verify_user` exchange. Change `LOGIN_HINT`/`LOGIN_ID`/`ACTION`/`SCOPE`/`OTP_SYSTEM` below to try a different login variant (see the findings cell for the other three already tried).

Running this cell sends a real OTP to the account's registered mobile number.

In [ ]:
ACTION = "profile/login"
SCOPE = ["abha-login", "mobile-verify"]
LOGIN_HINT = "abha-number"
LOGIN_ID_RAW = "91-4665-3075-0069"  # ABHA Number for poojaanchaliya@sbx (Pooja Rameshkumar)
OTP_SYSTEM = "abdm"

public_key = get_public_certificate()

otp_request_response = request_otp(
    action=ACTION,
    scope=SCOPE,
    login_hint=LOGIN_HINT,
    login_id=encrypt_value(LOGIN_ID_RAW, public_key),
    otp_system=OTP_SYSTEM,
)
print("status:", otp_request_response.status_code)
_otp_body = otp_request_response.json()
print(json.dumps(_otp_body, indent=2))

txn_id = _otp_body.get("txnId")
print("\ntxn_id:", txn_id)

## Step 3 — Verify OTP

`input()` blocks until you type the OTP you received and press Enter -- works fine in a running Jupyter kernel.

In [ ]:
otp_value = input("Enter the OTP you received: ").strip()

verify_response = verify_otp(
    action=ACTION,
    scope=SCOPE,
    txn_id=txn_id,
    otp_value=encrypt_value(otp_value, public_key),
)
print("status:", verify_response.status_code)
_verify_body = verify_response.json()
print(json.dumps(_verify_body, indent=2))

patient_token = _verify_body.get("token") or (_verify_body.get("tokens") or {}).get("token")
print("\npatient_token present:", bool(patient_token))
# Deliberately not printing patient_token itself -- it's a live bearer credential.

**Note:** if you change `LOGIN_HINT` above to `"mobile"` under `ACTION="profile/login"`, this step's `token` is a short-lived **T-Token**, not directly usable -- it needs an extra account-selection + `verify_user()` exchange first (see `server.abha.verify_user` and `tools/m1_test_suite/flows/login_mobile.py` for the reference implementation). The default `loginHint="abha-number"` above does NOT have this extra step.

## Step 4 — Call the two locker endpoints

In [ ]:
def get_subscribed_lockers(include_inactive: bool = True) -> requests.Response:
    url = f"{HIECM_BASE_URL}/subscription-requests/v3/patients/lockers"
    headers = {
        "REQUEST-ID": generate_request_id(),
        "TIMESTAMP": generate_timestamp(),
        "Authorization": f"Bearer {gateway_token}",
        "X-AUTH-TOKEN": patient_token,
        "X-CM-ID": X_CM_ID,
    }
    params = {"includeInactive": str(include_inactive).lower()}
    response = requests.get(url, headers=headers, params=params, timeout=30)
    print("status:", response.status_code)
    try:
        print(json.dumps(response.json(), indent=2))
    except ValueError:
        print(response.text)
    return response


lockers_response = get_subscribed_lockers()

## Step 5 — Locker details by locker ID

Needs a real `lockerId` from Step 4's response (once that returns one). Paste it into `LOCKER_ID` below.

In [ ]:
def get_locker_details(locker_id: str) -> requests.Response:
    url = f"{HIECM_BASE_URL}/subscription-requests/v3/patients/lockers/{locker_id}"
    headers = {
        "REQUEST-ID": generate_request_id(),
        "TIMESTAMP": generate_timestamp(),
        "Authorization": f"Bearer {gateway_token}",
        "X-AUTH-TOKEN": patient_token,
        "X-CM-ID": X_CM_ID,
    }
    response = requests.get(url, headers=headers, timeout=30)
    print("status:", response.status_code)
    try:
        print(json.dumps(response.json(), indent=2))
    except ValueError:
        print(response.text)
    return response


LOCKER_ID = ""  # <-- paste a real lockerId from Step 4's response

if LOCKER_ID:
    locker_details_response = get_locker_details(LOCKER_ID)
else:
    print("LOCKER_ID not set -- skipping (nothing to look up until Step 4 returns a locker).")